In [ ]:
import csv
import re
from rdflib import Graph, Namespace, URIRef
from collections import defaultdict
import json
from urllib.parse import quote
import os

In [ ]:
doaj_csv = r""
scimago_csv = r""
scopus_ttl= r""
loc_align_folder= r""
json_file = r""

In [16]:
# SCIMAGO has CATEGORIES and AREAS. Categories correspond to SCOPUS narrower (see the mapping in the scopus.csv file), while AREAS correspond to the broader categories in SCOPUS. 
# Cleaning the categories in the SCIMAGO file, since we don't need the Q1, Q2, Q3, Q4 info for the mapping with SCOPUS + stripping any extra whitespace.
# I needed a list of all the categories to check if they really align with the ones in SCOPUS (from the scopus.ttl file). 
# It looks like they do, but some cat. and areas are not ordered (e.g. the first Category in SCIMAGO should correspond to the fist Area and so on, but sometimes this is not the case.)

def clean_category(cat: str):
    return re.sub(r"\s*\(Q\d\)", "", cat).strip()

def map_areas_categories(input_csv: str):
    rows = set()

    with open(input_csv, encoding="utf-8") as infile:
        reader = csv.DictReader(infile, delimiter=";")

        for row in reader:
            areas = [a.strip() for a in row.get("Areas", "").split(";") if a.strip()]
            categories = [clean_category(c) for c in row.get("Categories", "").split(";") if c.strip()]

            if not areas or not categories:
                continue

            if len(areas) == 1:
                for c in categories:
                    rows.add((areas[0], c))
            else:
                for area, category in zip(areas, categories):
                    rows.add((area, category))

    # alphabetical order to better comare with the file scopus.csv
    rows = sorted(rows, key=lambda x: x[0])

    for area, category in rows:
        print(f"{area};{category}")

map_areas_categories(scimago_csv)

Agricultural and Biological Sciences;Insect Science
Agricultural and Biological Sciences;Ecology, Evolution, Behavior and Systematics
Agricultural and Biological Sciences;Cultural Studies
Agricultural and Biological Sciences;Biotechnology
Agricultural and Biological Sciences;Biochemistry, Genetics and Molecular Biology (miscellaneous)
Agricultural and Biological Sciences;Aquatic Science
Agricultural and Biological Sciences;Agronomy and Crop Science
Agricultural and Biological Sciences;Agricultural and Biological Sciences (miscellaneous)
Agricultural and Biological Sciences;Horticulture
Agricultural and Biological Sciences;Economics, Econometrics and Finance (miscellaneous)
Agricultural and Biological Sciences;History and Philosophy of Science
Agricultural and Biological Sciences;Soil Science
Agricultural and Biological Sciences;Ecology
Agricultural and Biological Sciences;Forestry
Agricultural and Biological Sciences;Animal Science and Zoology
Agricultural and Biological Sciences;Plant

In [17]:
# let's see how the Subjects are formatted in the DOAJ CSV file, to understand how to extract them

def doaj_cats(doaj_csv):
    with open(doaj_csv, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)

        for i, row in enumerate(reader):
            print(row["Subjects"])

            if i == 49:   # prime 50 righe
                break

doaj_cats(doaj_csv)
"""
    Dalle labels delle LOC categories nel JSON si vede che usa "--" come separatore tra categoria e sotto-categoria
    (es. "Philosophy. Psychology. Religion--Philosophy (General)"),
    mentre il DOAJ usa ":" (es. "Philosophy. Psychology. Religion: Philosophy (General)").
"""

Language and Literature: Greek language and literature. Latin language and literature
Law: Law of nations | Law: Law in general. Comparative and uniform law. Jurisprudence
Language and Literature: English language | Language and Literature: English literature
Language and Literature: Philology. Linguistics
Philosophy. Psychology. Religion: Philosophy (General)
Language and Literature: Philology. Linguistics
Medicine: Medicine (General)
Social Sciences: Commerce: Business: Accounting. Bookkeeping
Technology: Technology (General): Industrial engineering. Management engineering: Information technology
Geography. Anthropology. Recreation: Human ecology. Anthropogeography
Agriculture: Agriculture (General)
Law: Islamic law
Social Sciences
Philosophy. Psychology. Religion: Islam. Bahai Faith. Theosophy, etc.: Islam
Language and Literature: Philology. Linguistics
Social Sciences: Industries. Land use. Labor: Labor. Work. Working class | Social Sciences: Sociology (General)
Agriculture: Animal

'\n    Dalle labels delle LOC categories nel JSON si vede che usa "--" come separatore tra categoria e sotto-categoria\n    (es. "Philosophy. Psychology. Religion--Philosophy (General)"),\n    mentre il DOAJ usa ":" (es. "Philosophy. Psychology. Religion: Philosophy (General)").\n'

The following script parses and merges two different resources: the [text](<categories/File skos.ttl vocabolari/vocabolario_SCOPUS_validated.ttl>) and a folder containing LOC-SCOPUS alignment files in RDF/XML format (`.rdf`).

First, the Scopus taxonomy is parsed to extract each Scopus concept URI, its preferred label, broader areas, and narrower categories. The script builds an internal graph structure where every Scopus URI is associated with:

* a preferred label,
* related areas,
* related categories,
* labels for areas and categories.

Then, the script scans a folder containing LOC alignment RDF files. For each LOC concept (every file -> a LOC concept), it extracts:

* the LOC identifier,
* the preferred label,
* alternative labels (`skos:altLabel` and `skosxl:altLabel`),
* semantic alignments toward Scopus concepts (`closeMatch`, `broadMatch`, `narrowMatch`, `relatedMatch`).

During the merge phase, every Scopus alignment found in the LOC files is enriched with the information extracted from the Scopus taxonomy.

The script takes:

* one Scopus Turtle file (`scopus_ttl`)
* one folder containing RDF alignment files (`loc_align_folder`)

and produces a final JSON file (`merged_loc_scopus.json`).

The output structure is organized as a list of LOC entries. Each entry contains:

```json
{
  "loc_id": "...",
  "label": "...",
  "alt_labels": [],
  "scopus_alignments": [
    {
      "uri": "...",
      "type": "closeMatch",
      "label": "...",
      "area": [],
      "area_labels": [],
      "categories": [],
      "category_labels": []
    }
  ]
}
```

An example use case is the enrichment of SCImago journal data. For instance, a journal record such as:

```text
100;21101044878;"Minerva Cardiology and Angiology";journal;"27245683, 27245772";"Edizioni Minerva Medica S.p.A.";No;No;0,394;Q3;31;86;249;3468;281;207;1,24;40,33;37,63;0;Italy;Western Europe;"Edizioni Minerva Medica S.p.A.";"2021-2025";"Cardiology and Cardiovascular Medicine (Q3)";"Medicine"
```

can be connected to the enriched Scopus classification produced by the script. In this way, the journal subject category (`Cardiology and Cardiovascular Medicine`) can be linked to standardized Scopus concepts and find the corresponding LOC URI to align.

In [ ]:

SKOS = Namespace("http://www.w3.org/2004/02/skos/core#")
SKOSXL = Namespace("http://www.w3.org/2008/05/skos-xl#")

SCOPUS_NS = "https://www.scopus.com/vocabs/"
LOC_NS = "http://id.loc.gov/authorities/classification/"

# Utility functions

def is_scopus(uri):
    return str(uri).startswith(SCOPUS_NS)


def is_loc(uri):
    return str(uri).startswith(LOC_NS)


def create_safe_uri(uri):
    cleaned = uri.replace("\\(", "(").replace("\\)", ")") #RDF files were corrupted: I manually corrected them, but I automatically handled the escaping
    encoded = quote(cleaned, safe=":/#_-")
    return URIRef(encoded)


def clean_rdf_text(text):
    return text.replace("\\(", "(").replace("\\)", ")")

# Parse 'vocabolario_SCOPUS_validated.ttl


def parse_scopus_ttl(ttl_file):
    g = Graph()
    g.parse(ttl_file, format="turtle")

    graph = defaultdict(lambda: {
        "label": None,
        "area": set(),
        "categories": set(),
        "area_labels": set(),
        "category_labels": set(),
    })

    for s in g.subjects():

        if not is_scopus(s):
            continue

        s_str = str(s)

        label = g.value(s, SKOS.prefLabel)
        if label:
            graph[s_str]["label"] = str(label)

        # broader -> area
        for o in g.objects(s, SKOS.broader):

            if is_scopus(o):
                graph[s_str]["area"].add(str(o))

                o_label = g.value(o, SKOS.prefLabel)
                if o_label:
                    graph[s_str]["area_labels"].add(str(o_label))

        # inverse broader -> categories
        for o in g.subjects(SKOS.broader, s):

            if is_scopus(o):
                graph[s_str]["categories"].add(str(o))

                o_label = g.value(o, SKOS.prefLabel)
                if o_label:
                    graph[s_str]["category_labels"].add(str(o_label))

    return {
        uri: {
            "label": node["label"],
            "area": list(node["area"]),
            "categories": list(node["categories"]),
            "area_labels": list(node["area_labels"]),
            "category_labels": list(node["category_labels"]),
        }
        for uri, node in graph.items()
    }



# Parse single RDF alignment file in the "File LoC aggiornati con allineamento" folder


def parse_rdf_file(file_path):

    g = Graph()

    try:

        with open(file_path, "r", encoding="utf-8") as f:
            rdf_text = f.read()

        rdf_text = clean_rdf_text(rdf_text)

        g.parse(data=rdf_text, format="xml")

    except Exception as e:

        print(f"Skipped (corrupted): {os.path.basename(file_path)} - {e}")
        return []

    results = []

    for s in set(g.subjects()):

        if not is_loc(s):
            continue

        node = {
            "loc_id": str(s),
            "label": None,
            "alt_labels": [],
            "scopus_alignments": [],
        }

        # prefLabel
        label = g.value(s, SKOS.prefLabel)

        if label:
            node["label"] = str(label)

        # altLabel SKOSXL
        for alt in g.objects(s, SKOSXL.altLabel):

            literal = g.value(alt, SKOSXL.literalForm)

            if literal:
                node["alt_labels"].append(str(literal))

        # altLabel standard SKOS
        for alt in g.objects(s, SKOS.altLabel):
            node["alt_labels"].append(str(alt))

        # alignment predicates
        for pred, match_type in [
            (SKOS.closeMatch, "closeMatch"),
            (SKOS.narrowMatch, "narrowMatch"),
            (SKOS.broadMatch, "broadMatch"),
            (SKOS.relatedMatch, "relatedMatch"),
        ]:

            for o in g.objects(s, pred):

                if is_scopus(o):

                    node["scopus_alignments"].append({
                        "uri": str(create_safe_uri(str(o))),
                        "type": match_type,
                    })

        results.append(node)

    return results


# Parse "File LoC aggiornati con allineamento" folder

def parse_loc_folder(folder_path):
    all_data = []

    rdf_files = sorted(
        f for f in os.listdir(folder_path)
        if f.endswith(".rdf")
    )

    for filename in rdf_files:

        print(f"  Parsing: {filename}")

        file_path = os.path.join(folder_path, filename)

        all_data.extend(parse_rdf_file(file_path))

    return all_data


# Merge SCOPUS graph with LOC data 

def merge(loc_data, scopus_graph):

    merged = []
    missing_uris = []

    for loc_entry in loc_data:

        entry = dict(loc_entry)

        enriched_alignments = []

        for alignment in loc_entry.get("scopus_alignments", []):

            uri = alignment["uri"]

            info = scopus_graph.get(uri)

            enriched = {
                "uri": uri,
                "type": alignment["type"]
            }

            if info:

                enriched.update({
                    "label": info.get("label"),
                    "area": info.get("area", []),
                    "area_labels": info.get("area_labels", []),
                    "categories": info.get("categories", []),
                    "category_labels": info.get("category_labels", []),
                })

            else:

                enriched["warning"] = "URI not found in scopus_graph"
                missing_uris.append(uri)

            enriched_alignments.append(enriched)

        entry["scopus_alignments"] = enriched_alignments

        merged.append(entry)

    return merged, missing_uris



OUTPUT_JSON = "merged_loc_scopus.json"

scopus_graph = parse_scopus_ttl(scopus_ttl)

loc_data = parse_loc_folder(loc_align_folder)

merged_data, missing_uris = merge(loc_data, scopus_graph)

# SAVE

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:

    json.dump(
        merged_data,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nDone.")
print(f"Merged entries: {len(merged_data)}")
print(f"Missing URIs: {len(set(missing_uris))}")
print(f"Output saved to: {OUTPUT_JSON}")

Parsing Scopus TTL: C:\Users\ilari\Desktop\DHDK\OPEN SCIENCE\DATA\vocabolario_SCOPUS_validated.ttl
Parsing LOC alignment folder: C:\Users\ilari\Desktop\VS_CODE\OPEN SCIENCE\File LoC aggiornati con allineamento
  Parsing: AM1-AM501.skos.rdf
  Parsing: B1-B5802.skos.rdf
  Parsing: BF1-BF990.skos.rdf
  Parsing: BJ1-BJ1725.skos.rdf
  Parsing: BL1-BL2790.skos.rdf
  Parsing: CC1-CC960.skos.rdf
  Parsing: D1-D2027.skos.rdf
  Parsing: DA1-DA995.skos.rdf
  Parsing: DAW1001-DAW1051.skos.rdf
  Parsing: DB1-DB879.skos.rdf
  Parsing: DC1-DC947.skos.rdf
  Parsing: DE1-DE100.skos.rdf
  Parsing: DF10-DF951.skos.rdf
  Parsing: DG11-DG980.2.skos.rdf
  Parsing: DH1-DH207.skos.rdf
  Parsing: DJ1-DJ500.22.skos.rdf
  Parsing: DL1-DL1180.2.skos.rdf
  Parsing: DP1-DP402.skos.rdf
  Parsing: DQ1-DQ851.skos.rdf
  Parsing: DR1-DR2285.skos.rdf
  Parsing: DS1-DS937.skos.rdf
  Parsing: DT1-DT3415.skos.rdf
  Parsing: DU1-DU950.skos.rdf
  Parsing: DX101-DX301.skos.rdf
  Parsing: E11-E143.skos.rdf
  Parsing: F1-F975.sk

In [19]:
# Match DOAJ subjects with LOC alternative labels in order to assign LOC URIs.
# Both "--" and ": " forms are considered to improve matching compatibility.

def load_loc_mapping(json_file):

    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    mapping = {}

    for item in data:
        loc_id = item.get("loc_id")

        for alt in item.get("alt_labels", []):

            if not alt:
                continue

            # Original LOC form
            key_original = alt.strip().lower()
            mapping[key_original] = loc_id

            # DOAJ-compatible form
            key_colon = key_original.replace("--", ": ")
            mapping[key_colon] = loc_id

    return mapping


def parse_doaj_subjects(subjects_raw):
    """
    Extract candidate subjects from a DOAJ subject string.

    Rules:
      - "|" separates groups
      - ":" separates hierarchical levels
      - "." separates sub-subjects
    """

    if not subjects_raw:
        return []

    candidates = []

    for part in subjects_raw.split("|"):

        part = part.strip()

        # Hierarchical split
        tokens = [t.strip() for t in part.split(":")]

        for i, token in enumerate(tokens):

            # Extract sub-subjects
            for sub in token.split("."):

                sub = sub.strip()

                if sub:
                    candidates.append(sub)

            # Build progressive hierarchy
            if i > 0:
                compound = ": ".join(
                    t.strip() for t in part.split(":")[:i + 1]
                )
                candidates.append(compound)

        # Add full segment
        candidates.append(part)

    # Deduplicate while preserving order
    seen = set()
    result = []

    for c in candidates:

        key = c.lower()

        if key not in seen:
            seen.add(key)
            result.append(c)

    return result


def convert_doaj_to_json(doaj_csv, loc_json, output_json, N=None):
    """
    Convert DOAJ CSV data into structured JSON enriched with LOC URIs.

    For each DOAJ subject, all extracted candidates are tested
    against the LOC mapping. The first matching LOC URI is assigned.
    """

    label_to_loc = load_loc_mapping(loc_json)

    results = []

    with open(doaj_csv, "r", encoding="utf-8") as f:

        reader = csv.DictReader(f)

        for i, row in enumerate(reader):

            if N is not None and i >= N:
                break

            title = row.get("Journal title", "")
            issn = row.get("Journal ISSN (print version)", "")
            eissn = row.get("Journal EISSN (online version)", "")
            subjects_raw = row.get("Subjects", "")

            subjects_list = []

            if subjects_raw:

                # Extract visible DOAJ subjects
                surface_subjects = []

                for part in subjects_raw.split("|"):

                    part = part.strip()

                    tokens = [t.strip() for t in part.split(":")]

                    category = tokens[0]

                    if category:
                        surface_subjects.append(category)

                    if len(tokens) > 1:

                        last = tokens[-1]

                        for sub in last.split("."):

                            sub = sub.strip()

                            if sub:
                                surface_subjects.append(sub)

                # Deduplicate subjects
                seen = set()
                deduped = []

                for s in surface_subjects:

                    if s.lower() not in seen:
                        seen.add(s.lower())
                        deduped.append(s)

                for subject in deduped:

                    # Find matching LOC URI
                    candidates = parse_doaj_subjects(subject)

                    loc_uri = ""

                    for candidate in candidates:

                        loc_uri = label_to_loc.get(
                            candidate.lower(),
                            ""
                        )

                        if loc_uri:
                            break

                    subjects_list.append({
                        "subject": subject,
                        "loc_uri": loc_uri
                    })

            results.append({
                "journal_title": title,
                "journal_issn": issn,
                "journal_eissn": eissn,
                "subjects": subjects_list
            })

    with open(output_json, "w", encoding="utf-8") as f:

        json.dump(
            results,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(f"JSON saved to: {output_json} ({len(results)} records)")


convert_doaj_to_json(
    doaj_csv=doaj_csv,
    loc_json=json_file,
    output_json="doaj_journals_loc.json"
)

JSON saved to: doaj_journals_loc.json (22699 records)
